# Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from scipy.fft import fft, ifft, fftfreq

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.arima.model import ARIMA

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input, Conv1D, MaxPooling1D, Flatten, Dropout
from tensorflow.keras.optimizers import Adam

import os
import re
import json 
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def plot_figs (result, link, N, N_remove, save_figures, save_path, show_figs):
    # Plotting
    plt.figure(figsize=(18, 30))

    min_y = min(min(result[link]['Future_data']),
            min(result[link]['Future_estimated_data']),
            min(result[link]['Future_estimation_error']),
            min(result[link]['Future_data'] + result[link]['Future_mean_estimation_error']),
            min(result[link]['Future_data'] - result[link]['Future_mean_estimation_error']))

    max_y = max(max(result[link]['Future_data']),
                max(result[link]['Future_estimated_data']),
                max(result[link]['Future_estimation_error']),
                max(result[link]['Future_data'] + result[link]['Future_mean_estimation_error']),
                max(result[link]['Future_data'] - result[link]['Future_mean_estimation_error']))

    # Plot 1: Actual vs Estimated Data
    plt.subplot(4, 1, 2)
    plt.plot(result[link]['Future_dates'], result[link]['Future_data'], label='Actual Data', color='royalblue', marker='o')
    plt.plot(result[link]['Future_dates'], result[link]['Future_estimated_data'], label='Estimated Data', color='darkorange', linestyle='--', marker='x')
    plt.title(f'Actual vs Estimated Data for the future {N_remove} Days')
    plt.xlabel('Date')
    plt.ylabel('Error')
    plt.legend()
    plt.ylim(min_y, max_y)
    plt.grid()

    # Plot 2: Future Error Estimation Plot
    plt.subplot(4, 1, 3)
    plt.plot(result[link]['Future_dates'], result[link]['Future_estimation_error'], label='Error of the Estimation', color='green', marker='o')
    plt.axhline(result[link]['Future_mean_estimation_error'], linestyle='dashed', label='Average Estimation Error', color='purple')
    plt.axhline(result[link]['Future_rmse_estimation_error'], linestyle='dashed', label='Root Mean Squared Error', color='deeppink')
    plt.title(f'Prediction Error of Actual vs Estimated Data for the future {N_remove} Days     Avg: {result[link]["Future_mean_estimation_error"]:.5f} RMSE:{result[link]["Future_rmse_estimation_error"]:.5f}')
    plt.xlabel('Date')
    plt.ylabel('Error')
    plt.legend()
    plt.ylim(min_y, max_y)
    plt.grid()

    # Plot 3: Future Upper and Lower Bounds
    plt.subplot(4, 1, 4)
    plt.plot(result[link]['Future_dates'], result[link]['Future_data']+result[link]['Future_mean_estimation_error'], label='Upper Data Bound', color='teal', marker='o')
    plt.plot(result[link]['Future_dates'], result[link]['Future_data']-result[link]['Future_mean_estimation_error'], label='Lower Data Bound', color='orchid', marker='o')
    plt.plot(result[link]['Future_dates'], result[link]['Future_estimated_data'], label='Estimated Data', color='chocolate', linestyle='--', marker='x')
    plt.title(f'Bounds of Actual vs Estimated Data for the future {N_remove} Days   std: {result[link]["Future_std_estimation_error"]}')
    plt.xlabel('Date')
    plt.ylabel('Error')
    plt.legend()
    plt.ylim(min_y, max_y)
    plt.grid()


    plt.tight_layout()
    if save_figures:
        plt.savefig(save_path)
        plt.close()
    
    if show_figs: 
        plt.show()


# FFT

In [ ]:
def fft_estimator(data, N_remove, N_estimate, link, normalize_data, save_figures=False, save_path=None, show_figs = False):
    print(f'Estimating for link {link} (FFT)')
    # Extract the error values
    dates = data['Date']

    if normalize_data:
        max_value = max(data['Error'].values)
        data_values = np.floor((data['Error'].values /max_value) * 100)
    else:
        data_values = data['Error'].values 
   
    # Number of samples
    N = len(data_values)
    

    # Split the data
    if N_remove == 0:
        past_data_values = data_values
        past_dates = dates
        last_date = past_dates.iloc[-1]
        future_data_values =  pd.date_range(start=last_date, periods=N_estimate + 1, freq='D')[1:]
    else: 
        past_data_values = data_values[:-N_remove] # All the error values except for the last N_remove days
        future_data_values = data_values[-N_remove:] # The last N_remove error values
        past_dates = dates[:-N_remove] # All the error values except for the last N_remove days

   
    # Sample spacing (assuming daily data)
    T = 1.0

    # Apply the Fourier transform to the training data
    yf = fft(past_data_values)
    xf = fftfreq(len(past_data_values), T) # Computes the corresponding frequency bins

    # Remove outliers
    magnitude_threshold = np.percentile(np.abs(yf), 90)
    filtered_yf = yf * (np.abs(yf) > magnitude_threshold)

    # Estimate errors
    extended_yf_past = np.zeros(len(past_data_values) + N_remove, dtype=complex)
    extended_yf_past[:len(past_data_values)] = filtered_yf
    estimated_data = ifft(extended_yf_past).real
    estimated_data_past = estimated_data[:-N_remove]
    estimated_data_future = estimated_data[len(past_data_values):]

    # Generate future dates
    last_date = past_dates.iloc[-1]
    future_dates = pd.date_range(start=last_date, periods=N_estimate + 1, freq='D')[1:]

    # Calculate error metrics of the entire dataset
    error_of_fft = abs(data_values - estimated_data)
    mean_fft_error = np.mean(error_of_fft)
    std_fft_error = np.std(error_of_fft)
    rmse_fft_error = root_mean_squared_error(data_values, estimated_data)

    # Calculate error metrics of the future estimation
    error_of_future_estimation = abs(future_data_values - estimated_data_future)
    mean_error_future_estimation = np.mean(error_of_future_estimation)
    std_error_future_estimation = np.std(error_of_future_estimation)
    rmse_error_future_estimation = root_mean_squared_error(future_data_values, estimated_data_future)

    # Calculate error metrics of the past estimation
    error_of_past_estimation = abs(past_data_values - estimated_data_past)
    mean_error_past_estimation = np.mean(error_of_past_estimation)
    std_error_past_estimation = np.std(error_of_past_estimation)
    rmse_error_past_estimation = root_mean_squared_error(past_data_values, estimated_data_past)

    result = {
        link: {
            "Normalized_data": normalize_data,

            "Complete_dates": dates,
            "Complete_data": data_values,
            "Complete_estimated_data": estimated_data,

            "Complete_estimation_error": error_of_fft,
            "Complete_mean_estimation_error": mean_fft_error,
            "Complete_std_estimation_error": std_fft_error,
            "Complete_rmse_estimation_error": rmse_fft_error,

            
            "Past_dates": past_dates,
            "Last_date": last_date,
            "Past_data": past_data_values,
            "Past_estimated_data": estimated_data_past,

            "Past_estimation_error": error_of_past_estimation,
            "Past_mean_estimation_error": mean_error_past_estimation,
            "Past_std_estimation_error": std_error_past_estimation,
            "Past_rmse_estimation_error": rmse_error_past_estimation,


            "Future_dates": future_dates,
            "Future_data": future_data_values,
            "Future_estimated_data": estimated_data_future,

            "Future_estimation_error": error_of_future_estimation,
            "Future_mean_estimation_error": mean_error_future_estimation,
            "Future_std_estimation_error": std_error_future_estimation,
            "Future_rmse_estimation_error": rmse_error_future_estimation
        }
    }

    if show_figs or save_figures:
         plot_figs (result, link, N, N_remove, save_figures, save_path, show_figs)

    result[link]["Complete_dates"] = result[link]["Complete_dates"].to_list()
    result[link]["Past_dates"] = result[link]["Past_dates"].to_list()
    result[link]["Future_dates"] = result[link]["Future_dates"].to_list()
    
    for key in result:
        result[key]['Complete_dates'] = [date.strftime('%Y-%m-%d') for date in result[key]['Complete_dates']]
        result[key]['Past_dates'] = [date.strftime('%Y-%m-%d') for date in result[key]['Past_dates']]
        result[key]['Future_dates'] = [date.strftime('%Y-%m-%d') for date in result[key]['Future_dates']]
        result[key]['Last_date'] = result[key]['Last_date'].strftime('%Y-%m-%d')

    result[link]["Complete_data"] = result[link]["Complete_data"].tolist()
    result[link]["Complete_estimated_data"] = result[link]["Complete_estimated_data"].tolist()
    result[link]["Complete_estimation_error"] = result[link]["Complete_estimation_error"].tolist()
    result[link]["Past_data"] = result[link]["Past_data"].tolist()
    result[link]["Past_estimated_data"] = result[link]["Past_estimated_data"].tolist()
    result[link]["Past_estimation_error"] = result[link]["Past_estimation_error"].tolist()
    result[link]["Future_data"] = result[link]["Future_data"].tolist()
    result[link]["Future_estimated_data"] = result[link]["Future_estimated_data"].tolist()
    result[link]["Future_estimation_error"] = result[link]["Future_estimation_error"].tolist()

    with open(f"rebuttal/fft/fft_res_{link}.json", "w") as outfile: 
        json.dump(result, outfile)
        
    return result

In [ ]:
# # file_path = 'up_to_15-07-2024/brisbane_10-11.xlsx'
# df = pd.read_excel(file_path)
# fft_estimator(df, 30, 30, '101-102', False, save_figures=False, save_path=None, show_figs = True)

# RF

In [ ]:
def rf_estimator(data, days_to_remove, days_to_estimate, link, normalize_data, save_figures=False, save_path=None, show_figs = False):
    
    print(f'Estimating for link {link} (RF)')
        
    if normalize_data:
        max_value = max(data['Error'].values)
        data['Error'] = np.floor((data['Error'].values /max_value) * 100)
    # else:
    #     data['Error'] = data['Error'].values 

    data_df = data
    N = len(data['Error'])
    # Convert the date column to datetime
    data_df['Date'] = pd.to_datetime(data_df['Date'])

    # Extract the numerical representation of the date and additional features
    data_df['date_ordinal'] = data_df['Date'].map(datetime.toordinal)
    data_df['day_of_week'] = data_df['Date'].dt.dayofweek
    data_df['month'] = data['Date'].dt.month
    data_df['day_of_month'] = data_df['Date'].dt.day

    # Sort the dataframe by date
    data_df = data_df.sort_values(by='Date')

        # Split the data
    if days_to_remove == 0:
        past_df = data_df
        past_data_values = data['Error']
        past_dates = data['Date']
        last_date = past_dates.iloc[-1]
        future_data_values =  pd.date_range(start=last_date, periods=days_to_estimate + 1, freq='D')[1:]
        future_df = pd.DataFrame({'Date': future_data_values})
        # Adding additional features to future_df
        future_df['date_ordinal'] = future_df['Date'].map(datetime.toordinal)
        future_df['day_of_week'] = future_df['Date'].dt.dayofweek
        future_df['month'] = future_df['Date'].dt.month
        future_df['day_of_month'] = future_df['Date'].dt.day
        
    else: 
        # Separate the last 5 days from the dataset
        past_df = data_df.iloc[:-days_to_remove]
        future_df = data_df.iloc[-days_to_remove:]
        last_date = past_df['Date'].iloc[-1]

    # Prepare the features (X) and target (y)
    X = past_df[['date_ordinal', 'day_of_week', 'month', 'day_of_month']]
    y = past_df['Error']

    # Train the Random Forest model
        # n_estimators: Number of trees
        # criterion: Measure the quality of a split
    # model = RandomForestRegressor(n_estimators=100, criterion='squared_error', random_state=42)
    model = RandomForestRegressor(n_estimators=100, criterion='absolute_error', random_state=42)
    model.fit(X, y)

    # Prepare the features for the last 5 days
    X_future_pred = future_df[['date_ordinal', 'day_of_week', 'month', 'day_of_month']]
    y_future_actual = future_df['Error']

    # Predict fidelity for the last 5 days
    y_past_pred = model.predict(X)
    y_future_pred = model.predict(X_future_pred)

    #Metrics
        # Past
    error_of_past_estimation = abs(y - y_past_pred)
    mean_error_past_estimation = np.mean(error_of_past_estimation)
    std_error_past_estimation = np.std(error_of_past_estimation)
    rmse_error_past_estimation = root_mean_squared_error(y, y_past_pred)
        
        # Future
    error_of_future_estimation = abs(y_future_actual - y_future_pred)
    mean_error_future_estimation = np.mean(error_of_future_estimation)
    std_error_future_estimation = np.std(error_of_future_estimation)
    rmse_error_future_estimation = root_mean_squared_error(y_future_actual, y_future_pred)
        
        # Complete
    
    # complete_estimation = [ *y_past_pred, *y_future_pred]
    # error_of_complete_estimation = abs(data['Error'] - complete_estimation)
    # mean_error_complete_estimation = np.mean(error_of_complete_estimation)
    # std_error_complete_estimation = np.std(error_of_complete_estimation)
    # rmse_error_complete_estimation = root_mean_squared_error(data['Error'], complete_estimation)

    result = {
        link: {
            "Normalized_data": normalize_data,

            # "Complete_dates": data['Date'],
            # "Complete_data": data['Error'],
            # "Complete_estimated_data": complete_estimation,

            # "Complete_estimation_error": error_of_complete_estimation,
            # "Complete_mean_estimation_error": mean_error_complete_estimation,
            # "Complete_std_estimation_error": std_error_complete_estimation,
            # "Complete_rmse_estimation_error": rmse_error_complete_estimation,
                       
            "Past_dates": past_df['Date'],
            "Last_date": last_date,
            "Past_data": past_df['Error'],
            "Past_estimated_data": y_past_pred,

            "Past_estimation_error": error_of_past_estimation,
            "Past_mean_estimation_error": mean_error_past_estimation,
            "Past_std_estimation_error": std_error_past_estimation,
            "Past_rmse_estimation_error": rmse_error_past_estimation,


            "Future_dates": future_df['Date'],
            "Future_data": future_df['Error'],
            "Future_estimated_data": y_future_pred,

            "Future_estimation_error": error_of_future_estimation,
            "Future_mean_estimation_error": mean_error_future_estimation,
            "Future_std_estimation_error": std_error_future_estimation,
            "Future_rmse_estimation_error": rmse_error_future_estimation
        }
    }

    if show_figs or save_figures:
         plot_figs (result, link, N, days_to_remove, save_figures, save_path, show_figs)


    # result[link]["Complete_dates"] = result[link]["Complete_dates"].to_list()
    result[link]["Past_dates"] = result[link]["Past_dates"].to_list()
    result[link]["Future_dates"] = result[link]["Future_dates"].to_list()
    
    for key in result:
        # result[key]['Complete_dates'] = [date.strftime('%Y-%m-%d') for date in result[key]['Complete_dates']]
        result[key]['Past_dates'] = [date.strftime('%Y-%m-%d') for date in result[key]['Past_dates']]
        result[key]['Future_dates'] = [date.strftime('%Y-%m-%d') for date in result[key]['Future_dates']]
        result[key]['Last_date'] = result[key]['Last_date'].strftime('%Y-%m-%d')

    # result[link]["Complete_data"] = result[link]["Complete_data"].tolist()
    # result[link]["Complete_estimation_error"] = result[link]["Complete_estimation_error"].tolist()
    result[link]["Past_data"] = result[link]["Past_data"].tolist()
    result[link]["Past_estimated_data"] = result[link]["Past_estimated_data"].tolist()
    result[link]["Past_estimation_error"] = result[link]["Past_estimation_error"].tolist()
    result[link]["Future_data"] = result[link]["Future_data"].tolist()
    result[link]["Future_estimated_data"] = result[link]["Future_estimated_data"].tolist()
    result[link]["Future_estimation_error"] = result[link]["Future_estimation_error"].tolist()

    with open(f"rebuttal/rf/rf_res_{link}.json", "w") as outfile: 
        json.dump(result, outfile)

    return result

In [ ]:
# file_path = 'up_to_15-07-2024/brisbane_1-2.xlsx'
# df = pd.read_excel(file_path)
# a = rf_estimator(df, 30, 30, '101-102', False, save_figures=False, save_path=None, show_figs = True)

# LSTM

In [ ]:
def lstm_estimator(data, N_remove, N_estimate, link, normalize_data=True, epochs=50, save_figures=False, save_path=None, show_figs=False):
    print(f'Estimating for link {link} (LSTM)')

    # Data preparation as before
    dates = data['Date']
    data['DayOfWeek'] = data['Date'].dt.dayofweek
    data['Month'] = data['Date'].dt.month
    feature_columns = ['Error', 'DayOfWeek', 'Month']
    data_features = data[feature_columns]

    if normalize_data:
        scaler = MinMaxScaler(feature_range=(0, 1))
        data_values = scaler.fit_transform(data_features)
    else:
        data_values = data_features.values

    # Split the data
    N = len(data_values)
    if N_remove == 0:
        past_data_values = data_values
        past_dates = dates
        last_date = past_dates.iloc[-1]
        future_dates = pd.date_range(start=last_date, periods=N_estimate + 1, freq='D')[1:]
        future_data_values =  pd.date_range(start=last_date, periods=N_estimate + 1, freq='D')[1:]
        future_df = pd.DataFrame({'Date': future_data_values})
        # Adding additional features to future_df
        future_df['date_ordinal'] = future_df['Date'].map(datetime.toordinal)
        future_df['day_of_week'] = future_df['Date'].dt.dayofweek
        future_df['month'] = future_df['Date'].dt.month
        future_df['day_of_month'] = future_df['Date'].dt.day
        
    else:
        past_data_values = data_values[:-N_remove]
        future_data_values = data_values[-N_remove:]
        past_dates = dates[:-N_remove]
        future_dates = pd.date_range(start=dates.iloc[-N_remove], periods=N_estimate + 1, freq='D')[1:]
        last_date = past_dates.iloc[-1]
        
    # Create sequences for LSTM
    sequence_length = 60
    def create_sequences(data, sequence_length):
        sequences = []
        labels = []
        for i in range(sequence_length, len(data)):
            sequences.append(data[i-sequence_length:i])
            labels.append(data[i, 0])  # Predicting the 'Error' value (first column)
        return np.array(sequences), np.array(labels)
    
    X, y = create_sequences(past_data_values, sequence_length)
    # Split the data
    X_train, X_test = X[:-N_remove], X[-N_remove:]
    y_train, y_test = y[:-N_remove], y[-N_remove:]

    # Define the LSTM model using Input layer
    model = Sequential()
    
    # Add Input layer to define the input shape
    # model.add(Input(skhape=(X_train.shape[1], X_train.shape[2])))
    model.add(Input(shape=(X.shape[1], X.shape[2])))
    
    # Add LSTM layers
    model.add(LSTM(150, return_sequences=True))
    model.add(LSTM(150))
    model.add(Dense(1))

    # Compile the model
    model.compile(optimizer=Adam(learning_rate=0.001), loss='MeanSquaredError')

    # Train the model
    # history = model.fit(X_train, y_train, epochs=epochs, batch_size=32, verbose=0)
    history = model.fit(X, y, epochs=epochs, batch_size=32, verbose=0)

    # Predict past data
    # y_train_pred = model.predict(X_train)
    y_train_pred = model.predict(X)
    y_test_pred = model.predict(X_test)

    # Inverse transform if data was normalized
    if normalize_data:
        # y_train_pred = scaler.inverse_transform(np.concatenate([y_train_pred, X_train[:, -1, 1:]], axis=1))[:, 0]
        y_test_pred = scaler.inverse_transform(np.concatenate([y_test_pred, X_test[:, -1, 1:]], axis=1))[:, 0]
        # y_train = scaler.inverse_transform(np.concatenate([y_train.reshape(-1, 1), X_train[:, -1, 1:]], axis=1))[:, 0]
        y_test = scaler.inverse_transform(np.concatenate([y_test.reshape(-1, 1), X_test[:, -1, 1:]], axis=1))[:, 0]
        y_train_pred = scaler.inverse_transform(np.concatenate([y_train_pred, X[:, -1, 1:]], axis=1))[:, 0]
        y_train = scaler.inverse_transform(np.concatenate([y.reshape(-1, 1), X[:, -1, 1:]], axis=1))[:, 0]
    else:
        y_train_pred = y_train_pred
        y_test_pred = y_test_pred

    last_sequence = past_data_values[-sequence_length:].reshape((1, sequence_length, past_data_values.shape[1]))
    future_predictions = []

    for _ in range(N_estimate):
        future_pred = model.predict(last_sequence)[0]  # Make prediction

        if future_pred.size == 0:  # Check if prediction is valid
            break  # Avoid continuing if predictions are not generated

        future_predictions.append(future_pred)  # Append prediction to future_predictions

        # Update last_sequence for the next prediction step
        last_sequence = np.append(last_sequence[:, 1:, :], [[np.concatenate([future_pred, [last_sequence[0, -1, 1], last_sequence[0, -1, 2]]])]], axis=1)
        
    # Check if future_predictions contains data
    if len(future_predictions) == 0:
        raise ValueError("No future predictions were generated.")

    future_predictions = np.array(future_predictions).reshape(-1, 1)

    # Inverse transform future predictions if data was normalized
    if normalize_data:
        future_predictions = scaler.inverse_transform(np.concatenate([future_predictions, np.zeros((future_predictions.shape[0], 2))], axis=1))[:, 0]

    # Combine train and test predictions
    # complete_dates = pd.concat([pd.Series(dates[:-N_remove]), pd.Series(future_dates)])
    # complete_estimated_data = np.concatenate([model.predict(X).flatten(), future_predictions.flatten()])
    # complete_data = data['Error'][:-sequence_length]

    # Error metrics
    error_of_train_estimation = np.abs(y_train - y_train_pred)
    # mean_error_train_estimation = np.mean(error_of_train_estimation)
    # std_error_train_estimation = np.std(error_of_train_estimation)
    # # rmse_error_train_estimation = np.sqrt(mean_squared_error(y_train, y_train_pred))

    error_of_test_estimation = np.abs(y_test - y_test_pred)
    mean_error_test_estimation = np.mean(error_of_test_estimation)
    std_error_test_estimation = np.std(error_of_test_estimation)
    # rmse_error_test_estimation = np.sqrt(mean_squared_error(y_test, y_test_pred))
    
    result = {
        link: {
            "Normalized_data": normalize_data,

            # "Complete_dates": complete_dates,
            # "Complete_data": complete_data,
            # "Complete_estimated_data": complete_estimated_data.tolist(),
                       
            # "Past_dates": dates,
            # "Last_date": dates.iloc[-1],
            # "Past_data": data['Error'].values.tolist(),

            # "Train_dates": past_dates,
            # "Last_train_date": last_date,
            "Train_data": y_train.tolist(),
            # "Train_estimated_data": y_train_pred.tolist(),

            # "Train_estimation_error": error_of_train_estimation.tolist(),
            # "Train_mean_estimation_error": mean_error_train_estimation,
            # "Train_std_estimation_error": std_error_train_estimation,
            # "Train_rmse_estimation_error": rmse_error_train_estimation,

            # "Future_dates": future_df['Date'],
            # "Future_data": future_df['Error'],
            # "Future_estimated_data": future_predictions,

            # "Test_dates": future_dates,
            "Test_data": y_test.tolist(),
            "Test_estimated_data": future_predictions.tolist(),

            "Test_estimation_error": error_of_test_estimation.tolist(),
            "Test_mean_estimation_error": mean_error_test_estimation,
            "Test_std_estimation_error": std_error_test_estimation,
            # "Test_rmse_estimation_error": rmse_error_test_estimation,
        }
    }

    if show_figs or save_figures:
        plt.figure(figsize=(18, 30))

        # Train data plot
        plt.subplot(4, 1, 1)
        plt.plot(result[link]['Train_dates'][:len(result[link]['Train_data'])], result[link]['Train_data'], label='Actual Data', color='royalblue')
        plt.plot(result[link]['Train_dates'][:len(result[link]['Train_estimated_data'])], result[link]['Train_estimated_data'], label='Estimated Data', color='darkorange', linestyle='--')
        plt.title(f'LSTM\nActual vs Estimated Data (train)')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()

        # Test data plot
        plt.subplot(4, 1, 2)
        plt.plot(result[link]['Test_dates'][:len(result[link]['Test_data'])], result[link]['Test_data'], label='Actual Data', color='royalblue', marker='o')
        plt.plot(result[link]['Test_dates'][:len(result[link]['Test_estimated_data'])], result[link]['Test_estimated_data'], label='Estimated Data', color='darkorange', linestyle='--', marker='o')
        plt.title(f'Actual vs Estimated Data (test)  RMSE: {result[link]["Test_rmse_estimation_error"]}')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()

        # Upper and lower bounds for test data
        plt.subplot(4, 1, 3)
        plt.plot(result[link]['Test_dates'][:len(result[link]['Test_data'])], np.array(result[link]['Test_data']) + result[link]['Test_mean_estimation_error'], label='Upper Data Bound', color='teal', marker='o')
        plt.plot(result[link]['Test_dates'][:len(result[link]['Test_data'])], np.array(result[link]['Test_data']) - result[link]['Test_mean_estimation_error'], label='Lower Data Bound', color='orchid', marker='o')
        plt.plot(result[link]['Test_dates'][:len(result[link]['Test_estimated_data'])], result[link]['Test_estimated_data'], label='Estimated Data', color='chocolate', linestyle='--', marker='o')
        plt.title(f'Bounds of Actual vs Estimated Data (test) std: {result[link]["Test_std_estimation_error"]}')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()

        plt.tight_layout()
        if save_figures:
            plt.savefig(save_path)

        if show_figs:
            plt.show()

        plt.close()
    
    # result[link]["Complete_dates"] = result[link]["Complete_dates"].to_list()
    # result[link]["Train_dates"] = result[link]["Train_dates"].to_list()
    # result[link]["Test_dates"] = result[link]["Test_dates"].to_list()
    # result[link]["Past_dates"] = result[link]["Past_dates"].to_list()
    
    # for key in result:
    #     # result[key]['Complete_dates'] = [date.strftime('%Y-%m-%d') for date in result[key]['Complete_dates']]
    #     result[key]['Train_dates'] = [date.strftime('%Y-%m-%d') for date in result[key]['Train_dates']]
    #     result[key]['Test_dates'] = [date.strftime('%Y-%m-%d') for date in result[key]['Test_dates']]
        # result[key]['Past_dates'] = [date.strftime('%Y-%m-%d') for date in result[key]['Past_dates']]
        # result[key]['Last_date'] = result[key]['Last_date'].strftime('%Y-%m-%d')
        # result[key]['Last_train_date'] = result[key]['Last_train_date'].strftime('%Y-%m-%d')
        # result[key]['Future_dates'] = [date.strftime('%Y-%m-%d') for date in result[key]['Future_dates']]
        

    # result[link]["Test_estimated_data"] = result[link]["Test_estimated_data"].tolist()
    # result[link]["Test_data"] = result[link]["Test_data"].tolist()

    with open(f"rebuttal/lstm/lstm_res_{link}.json", "w") as outfile: 
        json.dump(result, outfile)

    return result

In [ ]:
# file_path = 'up_to_15-07-2024/brisbane_0-14.xlsx'
# df = pd.read_excel(file_path)
# a = lstm_estimator(df, 30, 30, '101-102', False, epochs=200, save_figures=False, save_path=None, show_figs = True)

# CNN

In [ ]:
def cnn_estimator(data, N_remove, N_estimate, link, normalize_data=True, epochs=50, batch_size=32, save_figures=False, save_path=None, show_figs=False):
    print(f'Estimating for link {link} (CNN)')

    # Extract the error values and new features
    dates = data['Date']
    # Feature engineering: Extract day of the week and month from the 'Date' column
    data['DayOfWeek'] = data['Date'].dt.dayofweek  # Monday=0, Sunday=6
    data['Month'] = data['Date'].dt.month  # Extracting month
    
    # Prepare the data for model input (including new features)
    feature_columns = ['Error', 'DayOfWeek', 'Month']
    data_features = data[feature_columns]

    # Normalize data if requested
    if normalize_data:
        scaler = MinMaxScaler(feature_range=(0, 1))
        data_values = scaler.fit_transform(data_features)
    else:
        data_values = data_features.values

    # Number of samples
    N = len(data_values)
    
    # Split the data
    if N_remove == 0:
        past_data_values = data_values
        past_dates = dates
        last_date = past_dates.iloc[-1]
        future_dates = pd.date_range(start=last_date, periods=N_estimate + 1, freq='D')[1:]
    else: 
        past_data_values = data_values[:-N_remove] # All the error values except for the last N_remove days
        future_data_values = data_values[-N_remove:] # The last N_remove error values
        past_dates = dates[:-N_remove] # All the error values except for the last N_remove days
        future_dates = pd.date_range(start=dates.iloc[-N_remove], periods=N_estimate + 1, freq='D')[1:]
        last_date = past_dates.iloc[-1]

    sequence_length = 10 
    # Create sequences for CNN (including all features)
    def create_sequences(data, sequence_length):
        sequences = []
        labels = []
        for i in range(sequence_length, len(data)):
            sequences.append(data[i-sequence_length:i])
            labels.append(data[i, 0])  # Predicting the 'Error' value (first column)
        return np.array(sequences), np.array(labels)
    
     # Number of past days to use to predict the next
    X, y = create_sequences(past_data_values, sequence_length)

    # Split into training and testing data
    X_train, X_test = X[:-N_remove], X[-N_remove:]
    y_train, y_test = y[:-N_remove], y[-N_remove:]

    # Reshape for CNN (samples, time steps, features)
    X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], X_train.shape[2]))
    X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], X_test.shape[2]))

    model = Sequential()

    # Add the Input layer with the input shape
    model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))

    # Add the rest of the CNN layers
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Conv1D(filters=128, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Flatten())
    model.add(Dense(100, activation='relu'))
    model.add(Dense(1))

    # Compile the model
    optimizer = Adam(learning_rate=0.001)
    # model.compile(optimizer=optimizer, loss='mean_squared_error')
    model.compile(optimizer=optimizer, loss='mean_absolute_error')

    # Train the model
    model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=0)

    # Predict past data
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Inverse transform if data was normalized
    if normalize_data:
        y_train_pred = scaler.inverse_transform(np.concatenate([y_train_pred, X_train[:, -1, 1:]], axis=1))[:, 0]
        y_test_pred = scaler.inverse_transform(np.concatenate([y_test_pred, X_test[:, -1, 1:]], axis=1))[:, 0]
        y_train = scaler.inverse_transform(np.concatenate([y_train.reshape(-1, 1), X_train[:, -1, 1:]], axis=1))[:, 0]
        y_test = scaler.inverse_transform(np.concatenate([y_test.reshape(-1, 1), X_test[:, -1, 1:]], axis=1))[:, 0]
    else:
        y_train_pred = y_train_pred
        y_test_pred = y_test_pred
    
    # Generate future predictions
    last_sequence = past_data_values[-sequence_length:].reshape((1, sequence_length, past_data_values.shape[1]))
    future_predictions = []
    
    for _ in range(N_estimate):
        future_pred = model.predict(last_sequence)[0]
        future_predictions.append(future_pred)
        last_sequence = np.append(last_sequence[:, 1:, :], [[np.concatenate([future_pred, [last_sequence[0, -1, 1], last_sequence[0, -1, 2]]])]], axis=1)
    
    future_predictions = np.array(future_predictions).reshape(-1, 1)
    
    # Inverse transform future predictions if data was normalized
    if normalize_data:
        future_predictions = scaler.inverse_transform(np.concatenate([future_predictions, np.zeros((future_predictions.shape[0], 2))], axis=1))[:, 0]

    # Combine train and test predictions
    complete_dates = pd.concat([pd.Series(dates[:-N_remove]), pd.Series(future_dates)])
    complete_estimated_data = np.concatenate([model.predict(X).flatten(), future_predictions.flatten()])

    # Error metrics
    error_of_train_estimation = np.abs(y_train - y_train_pred)
    mean_error_train_estimation = np.mean(error_of_train_estimation)
    std_error_train_estimation = np.std(error_of_train_estimation)
    rmse_error_train_estimation = np.sqrt(mean_squared_error(y_train, y_train_pred))

    error_of_test_estimation = np.abs(y_test - future_predictions)
    mean_error_test_estimation = np.mean(error_of_test_estimation)
    std_error_test_estimation = np.std(error_of_test_estimation)
    rmse_error_test_estimation = np.sqrt(mean_squared_error(y_test, future_predictions))

    result = {
        link: {
            "Normalized_data": normalize_data,

            "Complete_dates": complete_dates,
            "Complete_estimated_data": complete_estimated_data.tolist(),
                       
            "Past_dates": dates,
            "Last_date": dates.iloc[-1],
            "Past_data": data['Error'].values.tolist(),

            "Train_dates": past_dates,
            "Last_train_date": last_date,
            "Train_data": y_train.tolist(),
            "Train_estimated_data": y_train_pred.tolist(),

            "Train_estimation_error": error_of_train_estimation.tolist(),
            "Train_mean_estimation_error": mean_error_train_estimation,
            "Train_std_estimation_error": std_error_train_estimation,
            "Train_rmse_estimation_error": rmse_error_train_estimation,

            "Test_dates": future_dates,
            "Test_data": y_test.tolist(),
            "Test_estimated_data": future_predictions.tolist(),

            "Test_estimation_error": error_of_test_estimation.tolist(),
            "Test_mean_estimation_error": mean_error_test_estimation,
            "Test_std_estimation_error": std_error_test_estimation,
            "Test_rmse_estimation_error": rmse_error_test_estimation,
        }
    }

    # Plotting
    if show_figs or save_figures:
        plt.figure(figsize=(18, 30))

        train_data = np.array(result[link]['Train_data'])
        train_estimated_data = np.array(result[link]['Train_estimated_data'])
        test_data = np.array(result[link]['Test_data'])
        test_estimated_data = np.array(result[link]['Test_estimated_data'])
        test_upper_bound = test_data + result[link]['Test_mean_estimation_error']
        test_lower_bound = test_data - result[link]['Test_mean_estimation_error']
        test_estimation_error = np.array(result[link]['Test_estimation_error'])

        # Calculate min and max y-axis values for train and test data plots
        min_y = min(train_data.min(), train_estimated_data.min(), test_data.min(), test_estimated_data.min(), test_upper_bound.min(), test_lower_bound.min())
        max_y = max(train_data.max(), train_estimated_data.max(), test_data.max(), test_estimated_data.max(), test_upper_bound.max(), test_lower_bound.max())

        # Create the figure
        plt.figure(figsize=(18, 30))

        # Plot 1: Train Data
        plt.subplot(3, 1, 1)
        plt.plot(result[link]['Test_dates'][:len(test_data)], test_data, label='Actual Data', color='royalblue')
        plt.plot(result[link]['Test_dates'][:len(test_estimated_data)], test_estimated_data, label='Estimated Data', color='darkorange', linestyle='--')
        plt.title(f'LSTM\nActual vs Estimated Data (train)')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.ylim(min_y, max_y)
        plt.grid()

        # Plot 2: Test Data with Estimation Error
        plt.subplot(3, 1, 2)
        plt.plot(result[link]['Test_dates'][:len(test_estimation_error)], test_estimation_error[0], label='Error of the Estimation', color='green', marker='o')
        plt.axhline(result[link]['Test_mean_estimation_error'], linestyle='dashed', label='Average Estimation Error', color='purple')
        plt.axhline(result[link]['Test_rmse_estimation_error'], linestyle='dashed', label='Root Mean Squared Error', color='deeppink')
        plt.title(f'Prediction Error of Actual vs Estimated Data\n Avg: {result[link]["Test_mean_estimation_error"]:.5f} RMSE: {result[link]["Test_rmse_estimation_error"]:.5f}')
        plt.xlabel('Date')
        plt.ylabel('Estimation Error')
        plt.legend()
        plt.ylim(min_y, max_y)  # Use specific y-limits for the error plot
        plt.grid()

        # Plot 3: Upper and Lower Bounds for Test Data
        plt.subplot(3, 1, 3)
        plt.plot(result[link]['Test_dates'][:len(test_data)], test_upper_bound, label='Upper Data Bound', color='teal', marker='o')
        plt.plot(result[link]['Test_dates'][:len(test_data)], test_lower_bound, label='Lower Data Bound', color='orchid', marker='o')
        plt.plot(result[link]['Test_dates'][:len(test_estimated_data)], test_estimated_data, label='Estimated Data', color='chocolate', linestyle='--', marker='o')
        plt.title(f'Bounds of Actual vs Estimated Data (test) std: {result[link]["Test_std_estimation_error"]}')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.ylim(min_y, max_y)
        plt.grid()
        plt.tight_layout()
        if save_figures:
            plt.savefig(save_path)

        if show_figs:
            plt.show()

        plt.close()

    result[link]["Complete_dates"] = result[link]["Complete_dates"].to_list()
    result[link]["Train_dates"] = result[link]["Train_dates"].to_list()
    result[link]["Test_dates"] = result[link]["Test_dates"].to_list()
    result[link]["Past_dates"] = result[link]["Past_dates"].to_list()
    
    for key in result:
        result[key]['Complete_dates'] = [date.strftime('%Y-%m-%d') for date in result[key]['Complete_dates']]
        result[key]['Train_dates'] = [date.strftime('%Y-%m-%d') for date in result[key]['Train_dates']]
        result[key]['Test_dates'] = [date.strftime('%Y-%m-%d') for date in result[key]['Test_dates']]
        result[key]['Past_dates'] = [date.strftime('%Y-%m-%d') for date in result[key]['Past_dates']]
        result[key]['Last_date'] = result[key]['Last_date'].strftime('%Y-%m-%d')
        result[key]['Last_train_date'] = result[key]['Last_train_date'].strftime('%Y-%m-%d')

    with open(f"rebuttal/cnn/cnn_res_{link}.json", "w") as outfile: 
        json.dump(result, outfile)

    return result


In [ ]:
# file_path = 'up_to_15-07-2024/brisbane_0-14.xlsx'
# df = pd.read_excel(file_path)
# a = cnn_estimator(df, 30, 30, '101-102', False, epochs=250, save_figures=False, save_path=None, show_figs = True)

# ARIMA

In [ ]:
def arima_estimator(data, N_remove, N_estimate, link, normalize_data=True, save_figures=False, save_path=None, show_figs=False):
    print(f'Estimating for link {link} (ARIMA)')

    # Data preparation (use only the 'Error' column for ARIMA)
    dates = data['Date']
    target_column = 'Error'
    data_values = data[target_column].values
    
    if normalize_data:
        scaler = MinMaxScaler(feature_range=(0, 1))
        data_values = scaler.fit_transform(data[target_column].values.reshape(-1, 1)).flatten()
    else:
        data_values = data[target_column].values

    # Split the data into past and future
    if N_remove == 0:
        past_data_values = data_values
        last_date = dates.iloc[-1]
        future_dates = pd.date_range(start=last_date, periods=N_estimate + 1, freq='D')[1:]
    else:
        past_data_values = data_values[:-N_remove]
        future_data_values = data_values[-N_remove:]
        last_date = dates.iloc[-N_remove - 1]
        future_dates = pd.date_range(start=dates.iloc[-N_remove], periods=N_estimate + 1, freq='D')[1:]
    
    # Train ARIMA model on past data
    p, d, q = 10, 2, 1  # These are parameters of the ARIMA model, can be adjusted
    arima_model = ARIMA(past_data_values, order=(p, d, q))
    arima_model_fit = arima_model.fit()

    # Predict both past data and future data
    y_train_pred = arima_model_fit.predict(start=0, end=len(past_data_values)-1)
    future_predictions = arima_model_fit.forecast(steps=N_estimate)

    # Inverse transform if data was normalized
    if normalize_data:
        y_train_pred = scaler.inverse_transform(y_train_pred.reshape(-1, 1)).flatten()
        future_predictions = scaler.inverse_transform(future_predictions.reshape(-1, 1)).flatten()
        past_data_values = scaler.inverse_transform(past_data_values.reshape(-1, 1)).flatten()
        if N_remove != 0:
            future_data_values = scaler.inverse_transform(future_data_values.reshape(-1, 1)).flatten()

    # Error metrics for training data
    error_of_train_estimation = np.abs(past_data_values - y_train_pred)
    mean_error_train_estimation = np.mean(error_of_train_estimation)
    std_error_train_estimation = np.std(error_of_train_estimation)
    rmse_error_train_estimation = np.sqrt(mean_squared_error(past_data_values, y_train_pred))

    # If we have future data, calculate error metrics for future predictions
    if N_remove != 0:
        error_of_future_estimation = np.abs(future_data_values - future_predictions)
        mean_error_future_estimation = np.mean(error_of_future_estimation)
        std_error_future_estimation = np.std(error_of_future_estimation)
        rmse_error_future_estimation = np.sqrt(mean_squared_error(future_data_values, future_predictions))
    else:
        error_of_future_estimation = mean_error_future_estimation = std_error_future_estimation = rmse_error_future_estimation = None

    result = {
        link: {
            "Normalized_data": normalize_data,

            "Complete_dates": pd.concat([dates, pd.Series(future_dates)]),
            "Complete_data": np.concatenate([past_data_values, future_data_values]).tolist(),
            "Past_data": past_data_values.tolist(),
            "Past_dates": dates[:-N_remove],

            "Train_dates": dates[:-N_remove],
            "Last_train_date": last_date,
            "Train_data": past_data_values.tolist(),
            "Train_estimated_data": y_train_pred.tolist(),

            "Train_estimation_error": error_of_train_estimation.tolist(),
            "Train_mean_estimation_error": mean_error_train_estimation,
            "Train_std_estimation_error": std_error_train_estimation,
            "Train_rmse_estimation_error": rmse_error_train_estimation,

            "Test_dates": future_dates,
            "Test_data": future_data_values.tolist(),
            "Test_estimated_data": future_predictions.tolist(),

            # Future error metrics
            "Test_estimation_error": error_of_future_estimation.tolist(),
            "Test_mean_estimation_error": mean_error_future_estimation,
            "Test_std_estimation_error": std_error_future_estimation,
            "Test_rmse_estimation_error": rmse_error_future_estimation,
        }
    }


    if show_figs or save_figures:
        plt.figure(figsize=(18, 30))

        # Calculate the common Y-axis limits
        all_data = np.concatenate([
            result[link]['Train_data'],
            result[link]['Train_estimated_data'],
            result[link]['Test_data'],
            result[link]['Test_estimated_data'],
            np.array(result[link]['Test_data']) + result[link]['Test_mean_estimation_error'],
            np.array(result[link]['Test_data']) - result[link]['Test_mean_estimation_error']
        ])

        y_min, y_max = np.min(all_data)-0.0006, np.max(all_data)+0.0006

        # Train data plot
        plt.subplot(4, 1, 1)
        plt.plot(result[link]['Train_dates'][:len(result[link]['Train_data'])], result[link]['Train_data'], label='Actual Data', color='royalblue')
        plt.plot(result[link]['Train_dates'][:len(result[link]['Train_estimated_data'])], result[link]['Train_estimated_data'], label='Estimated Data', color='darkorange', linestyle='--')
        plt.title(f'CNN\nActual vs Estimated Data (train)')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()
        plt.ylim(y_min, y_max)  # Set common Y-axis limits

        # Test data plot
        plt.subplot(4, 1, 2)
        plt.plot(result[link]['Test_dates'][:len(result[link]['Test_data'])], result[link]['Test_data'], label='Actual Data', color='royalblue')
        plt.plot(result[link]['Test_dates'][:len(result[link]['Test_estimated_data'])], result[link]['Test_estimated_data'], label='Estimated Data', color='darkorange', linestyle='--')
        plt.title(f'Actual vs Estimated Data (test)  RMSE: {result[link]["Test_rmse_estimation_error"]}')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()
        plt.ylim(y_min, y_max)  # Set common Y-axis limits

        # Upper and lower bounds for test data
        plt.subplot(4, 1, 3)
        plt.plot(result[link]['Test_dates'][:len(result[link]['Test_data'])], np.array(result[link]['Test_data']) + result[link]['Test_mean_estimation_error'], label='Upper Data Bound', color='teal')
        plt.plot(result[link]['Test_dates'][:len(result[link]['Test_data'])], np.array(result[link]['Test_data']) - result[link]['Test_mean_estimation_error'], label='Lower Data Bound', color='orchid')
        plt.plot(result[link]['Test_dates'][:len(result[link]['Test_estimated_data'])], result[link]['Test_estimated_data'], label='Estimated Data', color='chocolate', linestyle='--', marker='o')
        plt.title(f'Bounds of Actual vs Estimated Data (test) std: {result[link]["Test_std_estimation_error"]}')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()
        plt.ylim(y_min, y_max)  # Set common Y-axis limits
    
        plt.tight_layout()
        if save_figures:
            plt.savefig(save_path)
            plt.close()
        
        if show_figs: 
            plt.show()
    

    result[link]["Train_dates"] = result[link]["Train_dates"].to_list()
    result[link]["Test_dates"] = result[link]["Test_dates"].to_list()
    result[link]["Past_dates"] = result[link]["Past_dates"].to_list()
    
    for key in result:
        result[key]['Complete_dates'] = [date.strftime('%Y-%m-%d') for date in result[key]['Complete_dates']]
        result[key]['Train_dates'] = [date.strftime('%Y-%m-%d') for date in result[key]['Train_dates']]
        result[key]['Test_dates'] = [date.strftime('%Y-%m-%d') for date in result[key]['Test_dates']]
        result[key]['Past_dates'] = [date.strftime('%Y-%m-%d') for date in result[key]['Past_dates']]
        result[key]['Last_train_date'] = result[key]['Last_train_date'].strftime('%Y-%m-%d')

    with open(f"rebuttal/arima/arima_res_{link}.json", "w") as outfile: 
        json.dump(result, outfile)

    return result


# Graphs

In [ ]:
def get_graphs(link, method): 
    # Load the JSON file
    file_path = f"rebuttal/{method}/{method}_res_{link}.json"
    with open(file_path, "r") as file:
        data = json.load(file)

    # Extract Train_data and Test_data
    # train_data = data[link]["Train_data"]
    # test_data = data[link]["Test_data"] 
    # test_estimated_data = data[link]["Test_estimated_data"]
    # estimation_error = data[link]["Test_estimation_error"]
    # mean_estimation_error = data[link]["Test_mean_estimation_error"]

    train_data = data[link]["Past_data"]
    test_data = data[link]["Future_data"] 
    test_estimated_data = data[link]["Future_estimated_data"]
    estimation_error = data[link]["Future_estimation_error"]
    mean_estimation_error = data[link]["Future_mean_estimation_error"]


    # Compute the average of Train_data
    train_data_avg = sum(train_data) / len(train_data)

    # Subtract train_data_avg from each value in Test_data
    adjusted_test_data = [abs(value - train_data_avg) for value in test_data]
    # adjusted_test_data = [(value - train_data_avg)**2 for value in test_data]

    mean_mean = sum(adjusted_test_data)/len(adjusted_test_data)
    # mean_mean = np.sqrt(mean_mean)
   

    # estimation_error = [(estimated - error)**2 for estimated, error in zip(test_estimated_data, test_data)]
    # mean_estimation_error = sum(estimation_error)/len(estimation_error)
    # mean_estimation_error = np.sqrt(mean_estimation_error)


    improvement = mean_estimation_error - mean_mean

    # Generate x-axis labels
    days = [f"Day {i+1}" for i in range(len(test_data))]
    
    # Plot the data
    plt.figure(figsize=(12, 6))

    #Data (marker: o, linestyle:--)
    plt.subplot(2,1,1)
    plt.plot(days, test_data, label='Actual Data', color='royalblue',  marker='o')
    plt.plot(days, test_estimated_data, label='Estimated Data (FFT)', color='darkorange', linestyle='--',  marker='o')
    plt.axhline(y=train_data_avg, label="Train Data Average", color='green', linestyle='--')
    plt.xlabel("Days")
    plt.ylabel("Qubit Pair Error")
    plt.xticks(rotation=45, fontsize=8)
    plt.legend()
    plt.grid(True)
    # plt.ylim(0.0, 0.0145)

    #Error
    plt.subplot(2,1,2)
    plt.plot(days, estimation_error, label="Error of the Estimation", linestyle="-.", color='darkorange', marker='x')
    plt.plot(days, adjusted_test_data, label="Error of the Average", linestyle="-.", color='green', marker='x')
    plt.axhline(y=mean_estimation_error, color='darkorange', linestyle="-.", label="Mean of the Estimation Error")
    plt.axhline(y=mean_mean, color='green', linestyle="-.", label="Mean of the Average Error")
    plt.xlabel("Days")
    plt.ylabel("Error")
    plt.xticks(rotation=45, fontsize=8)
    plt.legend()
    plt.grid(True)
    # plt.ylim(0.0, 0.0145)
    plt.tight_layout()

    plt.savefig(f'rebuttal/figs/{method}/{method}_{link}')
    plt.close()
    # Show plot
    # plt.show()
    
    return [link, mean_mean, mean_estimation_error, improvement]

In [ ]:
def get_graphs_no_avg(link, method): 
    # Load the JSON file
    file_path = f"rebuttal/{method}/{method}_res_{link}.json"
    with open(file_path, "r") as file:
        data = json.load(file)

    # Extract Train_data and Test_data
    train_data = data[link]["Train_data"]
    test_data = data[link]["Test_data"] 
    test_estimated_data = data[link]["Test_estimated_data"]
    estimation_error = data[link]["Test_estimation_error"]
    mean_estimation_error = data[link]["Test_mean_estimation_error"]

    # train_data = data[link]["Past_data"]
    # test_data = data[link]["Future_data"] 
    # test_estimated_data = data[link]["Future_estimated_data"]
    # estimation_error = data[link]["Future_estimation_error"]
    # mean_estimation_error = data[link]["Future_mean_estimation_error"]


    # Compute the average of Train_data
    train_data_avg = sum(train_data) / len(train_data)
    print(np.std(train_data))

    # Subtract train_data_avg from each value in Test_data
    adjusted_test_data = [abs(value - train_data_avg) for value in test_data]
    # adjusted_test_data = [(value - train_data_avg)**2 for value in test_data]

    mean_mean = sum(adjusted_test_data)/len(adjusted_test_data)
    # mean_mean = np.sqrt(mean_mean)
   

    # estimation_error = [(estimated - error)**2 for estimated, error in zip(test_estimated_data, test_data)]
    # mean_estimation_error = sum(estimation_error)/len(estimation_error)
    # mean_estimation_error = np.sqrt(mean_estimation_error)


    improvement = mean_estimation_error - mean_mean

    # Generate x-axis labels
    days = [f"Day {i+1}" for i in range(len(test_data))]
    
    # Plot the data
    plt.figure(figsize=(15, 5))

    #Data (marker: o, linestyle:--)
    plt.plot(days, test_data, label='Actual Data', color='royalblue',  marker='o')
    plt.plot(days, test_estimated_data, label='Estimated Data (ARIMA)', color='darkorange', linestyle='--',  marker='o')
    # plt.axhline(y=train_data_avg, label="Train Data Average", color='green', linestyle='--')
    # plt.plot(days, estimation_error, label="Error of the Estimation", linestyle="-.", color='darkorange', marker='x')
    # plt.plot(days, adjusted_test_data, label="Error of the Average", linestyle="-.", color='green', marker='x')
    plt.axhline(y=mean_estimation_error, color='purple', linestyle="-.", label="Mean of the Estimation Error")
    # plt.axhline(y=mean_mean, color='green', linestyle="-.", label="Mean of the Average Error")
    plt.xlabel("Days")
    plt.ylabel("Error")
    plt.xticks(rotation=45, fontsize=8)
    plt.legend()
    plt.grid(True)
    # plt.ylim(0.0, 0.0145)

    #Error
    # plt.subplot(2,1,2)
    # plt.plot(days, estimation_error, label="Error of the Estimation", linestyle="-.", color='darkorange', marker='x')
    # plt.plot(days, adjusted_test_data, label="Error of the Average", linestyle="-.", color='green', marker='x')
    # plt.axhline(y=mean_estimation_error, color='darkorange', linestyle="-.", label="Mean of the Estimation Error")
    # plt.axhline(y=mean_mean, color='green', linestyle="-.", label="Mean of the Average Error")
    # plt.xlabel("Days")
    # plt.ylabel("Error")
    # plt.xticks(rotation=45, fontsize=8)
    # plt.legend()
    # plt.grid(True)
    # # plt.ylim(0.0, 0.0145)
    plt.tight_layout()

    plt.savefig(f'rebuttal/figs/withoutAvg/{method}/{method}_{link}')
    plt.close()
    # Show plot
    # plt.show()
    
    # return [link, mean_mean, mean_estimation_error, improvement]

In [ ]:
def get_statistics(link, method): 
    # Load the JSON file
    file_path = f"rebuttal/{method}/{method}_res_{link}.json"
    with open(file_path, "r") as file:
        data = json.load(file)

    # Extract Train_data and Test_data
    train_data = data[link]["Train_data"]
    test_data = data[link]["Test_data"] 
    test_estimated_data = data[link]["Test_estimated_data"]
    estimation_error = data[link]["Test_estimation_error"]
    mean_estimation_error = data[link]["Test_mean_estimation_error"]

    # train_data = data[link]["Past_data"]
    # test_data = data[link]["Future_data"] 
    # test_estimated_data = data[link]["Future_estimated_data"]
    # estimation_error = data[link]["Future_estimation_error"]
    # mean_estimation_error = data[link]["Future_mean_estimation_error"]


    # Compute the average of Train_data
    train_data_avg = sum(train_data) / len(train_data)
    std_dev = np.std(train_data)

    # Subtract train_data_avg from each value in Test_data
    adjusted_test_data = [abs(value - train_data_avg) for value in test_data]
    # # adjusted_test_data = [(value - train_data_avg)**2 for value in test_data]

    # mean_mean = sum(adjusted_test_data)/len(adjusted_test_data)
    # # mean_mean = np.sqrt(mean_mean)
   

    # # estimation_error = [(estimated - error)**2 for estimated, error in zip(test_estimated_data, test_data)]
    # # mean_estimation_error = sum(estimation_error)/len(estimation_error)
    # # mean_estimation_error = np.sqrt(mean_estimation_error)


    # improvement = mean_estimation_error - mean_mean

    # # Generate x-axis labels
    # days = [f"Day {i+1}" for i in range(len(test_data))]
    
    # # Plot the data
    # plt.figure(figsize=(15, 5))

    # #Data (marker: o, linestyle:--)
    # plt.plot(days, test_data, label='Actual Data', color='royalblue',  marker='o')
    # plt.plot(days, test_estimated_data, label='Estimated Data (ARIMA)', color='darkorange', linestyle='--',  marker='o')
    # # plt.axhline(y=train_data_avg, label="Train Data Average", color='green', linestyle='--')
    # # plt.plot(days, estimation_error, label="Error of the Estimation", linestyle="-.", color='darkorange', marker='x')
    # # plt.plot(days, adjusted_test_data, label="Error of the Average", linestyle="-.", color='green', marker='x')
    # plt.axhline(y=mean_estimation_error, color='purple', linestyle="-.", label="Mean of the Estimation Error")
    # # plt.axhline(y=mean_mean, color='green', linestyle="-.", label="Mean of the Average Error")
    # plt.xlabel("Days")
    # plt.ylabel("Error")
    # plt.xticks(rotation=45, fontsize=8)
    # plt.legend()
    # plt.grid(True)
    # # plt.ylim(0.0, 0.0145)

    # #Error
    # # plt.subplot(2,1,2)
    # # plt.plot(days, estimation_error, label="Error of the Estimation", linestyle="-.", color='darkorange', marker='x')
    # # plt.plot(days, adjusted_test_data, label="Error of the Average", linestyle="-.", color='green', marker='x')
    # # plt.axhline(y=mean_estimation_error, color='darkorange', linestyle="-.", label="Mean of the Estimation Error")
    # # plt.axhline(y=mean_mean, color='green', linestyle="-.", label="Mean of the Average Error")
    # # plt.xlabel("Days")
    # # plt.ylabel("Error")
    # # plt.xticks(rotation=45, fontsize=8)
    # # plt.legend()
    # # plt.grid(True)
    # # # plt.ylim(0.0, 0.0145)
    # plt.tight_layout()

    # plt.savefig(f'rebuttal/figs/withoutAvg/{method}/{method}_{link}')
    # plt.close()
    # # Show plot
    # plt.show()
    
    # return [link, mean_mean, mean_estimation_error, improvement]

# Run

In [ ]:
pattern = r'_(\d{1,3}-\d{1,3})\.xlsx'
method = 'arima'

# files_in_dir = os.listdir('./LauraHandy/data/not_normalized/Brisbane/up_to_15-07-2024')
files_in_dir = os.listdir('C:/Users/lrodsor1.UPVNET/OneDrive - UPV/Documentos/Code/error_estimation/LauraHandy/data/not_normalized/Brisbane/up_to_15-07-2024')
# C:\Users\lrodsor1.UPVNET\OneDrive - UPV\Documentos\Code\error_estimation\LauraHandy\data\not_normalized\Brisbane\up_to_15-07-2024

df = pd.DataFrame(columns=['Link', 'Error_avg', 'Error_estimation', 'Improvement'])

data_files = []
for file_name in files_in_dir:
    data_files.append('C:/Users/lrodsor1.UPVNET/OneDrive - UPV/Documentos/Code/error_estimation/LauraHandy/data/not_normalized/Brisbane/up_to_15-07-2024/' + file_name)

for file_path in data_files:
    # data = pd.read_excel(file_path)
    link = re.search(pattern, file_path).group(1)
    # arima_estimator(data, 30, 30, link, normalize_data=True, save_figures=False, save_path=None, show_figs=False)
    # rf_estimator(data, 30, 30, link, False, False, None, False)
    # lstm_estimator(data, 30, 30, link, True, epochs=150, save_figures=False, save_path=None, show_figs=False)
    # cnn_estimator(data, 30, 30, link, normalize_data=True, epochs=250, batch_size=32, save_figures=False, save_path=None, show_figs=False)
    # fft_estimator(data, 30, 30, link, False, False, None, False)
    # result = get_graphs(link, method)
    get_graphs_no_avg(link, method)
    # print(result)
    # df.loc[len(df)] = result
# df.to_csv(f'rebuttal/estimation_improvement_{method}.csv', index=False)

In [ ]:
get_graphs_no_avg("49-50", 'arima')

# Other

In [ ]:
import time
days_to_remove = 30
days_to_estimate = 30
pattern = r'_(\d{1,3}-\d{1,3})\.xlsx'
# pattern = r'sherbrooke_(\d{1,3}-\d{1,3})_max_\d+\.\d+\.xlsx'
normalize_data = False
save_figures = False
show_figs = False

fft_result_dict = {}
rf_result_dict = {}
lstm_result_dict = {}
cnn_result_dict = {}
gbdt_result_dict = {}
arima_result_dict = {}

files_in_dir = os.listdir('up_to_04-10-24')
save_path = f'res/real_backend'
start_time = time.time()
data_files = []
for file_name in files_in_dir:
    data_files.append('up_to_04-10-24/' + file_name)
max_error = 0
for file_path in data_files:
    data = pd.read_excel(file_path)
    link = re.search(pattern, file_path).group(1)
    if max(data['Error']) > max_error:
        max_error = max(data['Error'])
    if normalize_data:
        save_path_fig = f'{save_path}Normalized/Link_{link}.png'
    else:
        save_path_fig = f'{save_path}/Not_normalized/Link_{link}.png'

    # rf_result_dict.update(rf_estimator(data, days_to_remove, days_to_estimate, link, normalize_data, save_figures, save_path_fig, show_figs))
    # fft_result_dict.update(fft_estimator(data, days_to_remove, days_to_estimate, link, normalize_data, save_figures, save_path_fig, show_figs))
    # lstm_result_dict.update(lstm_estimator(data, days_to_remove, days_to_estimate, link, normalize_data=True, epochs=150, save_figures=False, save_path=None, show_figs=False))
    cnn_result_dict.update(cnn_estimator(data, days_to_remove, days_to_estimate, link, normalize_data=True, epochs=250, batch_size=32, save_figures=False, save_path=None, show_figs=False))
    # gbdt_result_dict.update(gbdt_estimator(data, days_to_remove, days_to_estimate, link, normalize_data=True, epochs=100, batch_size=32, save_figures=False, save_path=None, show_figs=False))
    # arima_result_dict.update(arima_estimator(data, days_to_remove, days_to_estimate, link, normalize_data=True, save_figures=False, save_path=None, show_figs=False))

result_dict = {}  

# result_dict['FFT'] = fft_result_dict
# result_dict['RF'] = rf_result_dict  
# result_dict['LSTM'] = lstm_result_dict
result_dict['CNN'] = cnn_result_dict
# result_dict['GBDT'] = gbdt_result_dict
# result_dict['ARIMA'] = arima_result_dict

if normalize_data:
    save_path_json = f'{save_path}/results_7.json'
else:
    save_path_json = f'{save_path}/results_7.json'
print("--- %s seconds ---" % (time.time() - start_time))
# with open(save_path_json, "w") as outfile: 
#     json.dump(result_dict, outfile)


In [ ]:
for key in result_dict['LSTM']:
    # result_dict['LSTM'][key]['Future_dates'] = [date.strftime('%Y-%m-%d') for date in result_dict['LSTM'][key]['Future_dates']]
    result_dict['LSTM'][key]["Future_estimated_data"] = result_dict['LSTM'][key]["Future_estimated_data"].tolist()

In [ ]:
for key in result_dict['LSTM']['0-1']:
    print(type(result_dict['LSTM']['0-1'][key]))


In [ ]:
with open(save_path_json, "w") as outfile: 
    json.dump(result_dict, outfile)


In [ ]:
result_dict

# Mean

In [ ]:
pattern = r'_(\d{1,3}-\d{1,3})\.xlsx'
method = 'cnn'

# files_in_dir = os.listdir('./LauraHandy/data/not_normalized/Brisbane/up_to_15-07-2024')
files_in_dir = os.listdir('C:/Users/lrodsor1.UPVNET/OneDrive - UPV/Documentos/Code/error_estimation/LauraHandy/data/not_normalized/Brisbane/up_to_15-07-2024')
# C:\Users\lrodsor1.UPVNET\OneDrive - UPV\Documentos\Code\error_estimation\LauraHandy\data\not_normalized\Brisbane\up_to_15-07-2024

df = pd.DataFrame(columns=['Link', 'Error_avg'])

data_files = []
for file_name in files_in_dir:
    data_files.append('C:/Users/lrodsor1.UPVNET/OneDrive - UPV/Documentos/Code/error_estimation/LauraHandy/data/not_normalized/Brisbane/up_to_15-07-2024/' + file_name)

for file_path in data_files:
# for i in range(1):
    data = pd.read_excel(file_path)
    link = re.search(pattern, file_path).group(1)
    data_values = data['Error']
    data_values = data_values[:-30]
    avg_error = sum(data_values)/len(data_values)
    df.loc[len(df)] = [link, avg_error]
    
df[['Node1', 'Node2']] = df['Link'].str.split('-', expand=True).astype(int)

# Calculate the required value (1 - Error_avg)
df['Transformed_Value'] = 1 - df['Error_avg']

# Select the required columns
df_final = df[['Node1', 'Node2', 'Transformed_Value']]

# Save to a text file
file_path = "rebuttal//transformed_data.txt"
df_final.to_csv(file_path, sep=' ', index=False, header=False)
